In [ ]:
# Cell 1: Install required libraries
!pip install -q transformers torch scikit-learn pandas numpy matplotlib seaborn plotly wordcloud nltk streamlit pyngrok sentence-transformers faiss-cpu tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.9 MB/s eta 0:00:00


In [ ]:
# Cell 2: Import libraries and check GPU
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ Using device: cuda
GPU: Tesla T4
GPU Memory: 15.83 GB


In [ ]:
# Cell 3: Load your dataset (57K songs)
# Assuming your file is uploaded to Colab
df = pd.read_csv('/content/spotify_millsongdata.csv')  # Change to your actual filename

print("="*60)
print("📊 DATASET LOADED SUCCESSFULLY")
print("="*60)
print(f"Total songs: {len(df):,}")
print(f"Total artists: {df['artist'].nunique():,}")
print(f"Dataset shape: {df.shape}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

📊 DATASET LOADED SUCCESSFULLY
Total songs: 57,650
Total artists: 643
Dataset shape: (57650, 4)

Memory usage: 81.76 MB


In [ ]:
# Drop the link column as requested
df = df.drop('link', axis=1)
print(f"\n✅ Dropped 'link' column. New columns: {df.columns.tolist()}")


✅ Dropped 'link' column. New columns: ['artist', 'song', 'text']


In [ ]:
# Cell 4: Comprehensive EDA
print("="*60)
print("📈 EXPLORATORY DATA ANALYSIS (EDA)")
print("="*60)

# 1. Dataset sample
print("\n1️⃣ DATASET SAMPLE (First 5 songs):")
print("-"*60)
for i, row in df.head().iterrows():
    print(f"Song {i+1}: {row['song']} by {row['artist']}")
    print(f"   Lyrics preview: {row['text'][:100]}...")
    print()

# 2. Basic statistics
print("\n2️⃣ BASIC STATISTICS:")
print("-"*60)
print(df.describe(include='all'))

# 3. Missing values
print("\n3️⃣ MISSING VALUES:")
print("-"*60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

# 4. Text length analysis
df['lyrics_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print("\n4️⃣ LYRICS LENGTH STATISTICS:")
print("-"*60)
print(f"Average lyrics length: {df['lyrics_length'].mean():.0f} characters")
print(f"Average word count: {df['word_count'].mean():.0f} words")
print(f"Shortest lyrics: {df['lyrics_length'].min()} chars")
print(f"Longest lyrics: {df['lyrics_length'].max()} chars")

# 5. Top artists
print("\n5️⃣ TOP 10 ARTISTS:")
print("-"*60)
top_artists = df['artist'].value_counts().head(10)
for i, (artist, count) in enumerate(top_artists.items(), 1):
    print(f"{i:2d}. {artist}: {count:,} songs")

📈 EXPLORATORY DATA ANALYSIS (EDA)

1️⃣ DATASET SAMPLE (First 5 songs):
------------------------------------------------------------
Song 1: Ahe's My Kind Of Girl by ABBA
   Lyrics preview: Look at her face, it's a wonderful face  
And it means something special to me  
Look at the way t...

Song 2: Andante, Andante by ABBA
   Lyrics preview: Take it easy with me, please  
Touch me gently like a summer evening breeze  
Take your time, make...

Song 3: As Good As New by ABBA
   Lyrics preview: I'll never know why I had to go  
Why I had to put up such a lousy rotten show  
Boy, I was tough,...

Song 4: Bang by ABBA
   Lyrics preview: Making somebody happy is a question of give and take  
You can learn how to show it so come on, giv...

Song 5: Bang-A-Boomerang by ABBA
   Lyrics preview: Making somebody happy is a question of give and take  
You can learn how to show it so come on, giv...


2️⃣ BASIC STATISTICS:
------------------------------------------------------------
              ar

In [ ]:
# Cell 6: Preprocessing with NLTK
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added to download the missing resource

def preprocess_lyrics(text):
    """Clean and preprocess lyrics"""
    if not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove special characters but keep words
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    # Keep some emotion words
    keep_words = {'not', 'no', 'never', 'always', 'very', 'too', 'more', 'most', 'less', 'least'}
    stop_words = stop_words - keep_words
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)

print("🔄 Preprocessing lyrics...")
df['processed_text'] = df['text'].apply(preprocess_lyrics)
print(f"✅ Processed {len(df):,} songs")
print(f"\nSample before: {df['text'].iloc[0][:100]}...")
print(f"Sample after: {df['processed_text'].iloc[0][:100]}...")

🔄 Preprocessing lyrics...
✅ Processed 57,650 songs

Sample before: Look at her face, it's a wonderful face  
And it means something special to me  
Look at the way t...
Sample after: look face wonderful face mean something special look way smile see lucky one fellow kind girl make f...


In [ ]:
# Cell 7: Load BERT model on GPU
from sentence_transformers import SentenceTransformer

print("🔄 Loading BERT model on GPU...")
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
semantic_model = semantic_model.to(device)
print("✅ BERT model loaded on", device)

🔄 Loading BERT model on GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ BERT model loaded on cuda


In [ ]:
# Cell 8: Generate semantic embeddings on GPU
print(f"🔄 Generating semantic embeddings for {len(df):,} songs...")

def get_bert_embeddings_batch(texts, batch_size=128):
    """Generate BERT embeddings in batches"""
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Move to GPU for processing
        embeddings = semantic_model.encode(batch,
                                          show_progress_bar=False,
                                          convert_to_tensor=True,
                                          device=device)
        all_embeddings.append(embeddings.cpu().numpy())  # Move back to CPU for storage

    return np.vstack(all_embeddings)

# Get embeddings for all songs
texts = df['processed_text'].tolist()
bert_embeddings = get_bert_embeddings_batch(texts, batch_size=128)
print(f"✅ Semantic embeddings shape: {bert_embeddings.shape}")
print(f"   Memory: {bert_embeddings.nbytes / 1024**3:.2f} GB")

🔄 Generating semantic embeddings for 57,650 songs...
✅ Semantic embeddings shape: (57650, 384)
   Memory: 0.08 GB


In [ ]:
# Cell 9: Emotion analysis on GPU
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

print("🔄 Loading emotion model on GPU...")
emotion_model_name = "j-hartmann/emotion-english-distilroberta-base"
emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_name)
emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_name)
emotion_model = emotion_model.to(device)

# Create pipeline with GPU
emotion_pipeline = pipeline("text-classification",
                            model=emotion_model,
                            tokenizer=emotion_tokenizer,
                            return_all_scores=True,
                            device=0 if torch.cuda.is_available() else -1)

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

def get_emotion_vector(text, max_length=300):
    """Get emotion vector for text"""
    if len(text.split()) > max_length:
        text = ' '.join(text.split()[:max_length])

    try:
        results = emotion_pipeline(text, truncation=True)[0]
        scores = {item['label']: item['score'] for item in results}
        return np.array([scores.get(label, 0) for label in emotion_labels])
    except:
        return np.zeros(len(emotion_labels))

print("🔄 Generating emotion vectors...")
# Process in batches for efficiency
emotion_vectors = []
batch_size = 64

for i in range(0, len(df), batch_size):
    batch_texts = df['processed_text'].iloc[i:i+batch_size].tolist()
    batch_vectors = [get_emotion_vector(text) for text in batch_texts]
    emotion_vectors.extend(batch_vectors)

    if i % 5000 == 0:
        print(f"  Processed {min(i+batch_size, len(df)):,} / {len(df):,} songs")

emotion_embeddings = np.array(emotion_vectors)
print(f"✅ Emotion embeddings shape: {emotion_embeddings.shape}")

🔄 Loading emotion model on GPU...


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔄 Generating emotion vectors...


model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

  Processed 64 / 57,650 songs
  Processed 40,064 / 57,650 songs
✅ Emotion embeddings shape: (57650, 7)


In [ ]:
# Cell 10: Topic modeling with LDA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

print("🔄 Performing topic modeling...")

# Create TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000,
                             min_df=10,
                             max_df=0.7,
                             stop_words='english',
                             ngram_range=(1, 2))

tfidf_matrix = vectorizer.fit_transform(df['processed_text'].sample(50000, random_state=42))

# Apply LDA
n_topics = 15
lda = LatentDirichletAllocation(n_components=n_topics,
                                random_state=42,
                                learning_method='online',
                                n_jobs=-1,
                                verbose=1)

lda_topics_small = lda.fit_transform(tfidf_matrix)

# Now transform all songs
print("🔄 Transforming all songs to topic space...")
all_tfidf = vectorizer.transform(df['processed_text'])
topic_embeddings = lda.transform(all_tfidf)
print(f"✅ Topic embeddings shape: {topic_embeddings.shape}")

# Show topics
def get_topic_words(model, feature_names, n_words=15):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[:-n_words-1:-1]
        top_words = [feature_names[i] for i in top_words_idx]
        topics[f'Topic_{topic_idx+1}'] = top_words
    return topics

feature_names = vectorizer.get_feature_names_out()
topics = get_topic_words(lda, feature_names)
print("\n📚 DISCOVERED TOPICS:")
for topic, words in topics.items():
    print(f"{topic}: {', '.join(words[:8])}...")

🔄 Performing topic modeling...
iteration: 1 of max_iter: 10
iteration: 2 of max_iter: 10
iteration: 3 of max_iter: 10
iteration: 4 of max_iter: 10
iteration: 5 of max_iter: 10
iteration: 6 of max_iter: 10
iteration: 7 of max_iter: 10
iteration: 8 of max_iter: 10
iteration: 9 of max_iter: 10
iteration: 10 of max_iter: 10
🔄 Transforming all songs to topic space...
✅ Topic embeddings shape: (57650, 15)

📚 DISCOVERED TOPICS:
Topic_1: yeah yeah, yeah, dance, alright, hello, dance dance, alright alright, little bit...
Topic_2: la, la la, da, bye, da da, bye bye, mi, di...
Topic_3: oo, si, ooo, ni, bluebird, ne, en, che...
Topic_4: love, know, time, heart, want, say, oh, like...
Topic_5: ooh, ooh ooh, woman, mind mind, maria, hoo, whispering, love hear...
Topic_6: yang, mama, goodnight, boom, higher, crazy, woah, higher higher...
Topic_7: doo, doo doo, pum, noel, molly, pum pum, jah, pa rum...
Topic_8: gon na, gon, na, na gon, na make, know gon, shine, na love...
Topic_9: kindness, alive aliv

In [ ]:
# Cell 11: Create FAISS CPU index for fast similarity search
import faiss

print("🔄 Building FAISS CPU index...")

# Normalize embeddings for cosine similarity
def normalize_embeddings(embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norms

# Normalize all embeddings
bert_norm = normalize_embeddings(bert_embeddings.astype('float32'))
emotion_norm = normalize_embeddings(emotion_embeddings.astype('float32'))
topic_norm = normalize_embeddings(topic_embeddings.astype('float32'))

# Create indexes on CPU (since faiss-cpu is installed)
bert_index_cpu = faiss.IndexFlatIP(bert_norm.shape[1])
bert_index_cpu.add(bert_norm)

emotion_index_cpu = faiss.IndexFlatIP(emotion_norm.shape[1])
emotion_index_cpu.add(emotion_norm)

topic_index_cpu = faiss.IndexFlatIP(topic_norm.shape[1])
topic_index_cpu.add(topic_norm)

print(f"✅ Created 3 FAISS CPU indexes:")
print(f"   • Semantic index: {bert_index_cpu.ntotal:,} vectors")
print(f"   • Emotion index: {emotion_index_cpu.ntotal:,} vectors")
print(f"   • Topic index: {topic_index_cpu.ntotal:,} vectors")

🔄 Building FAISS CPU index...
✅ Created 3 FAISS CPU indexes:
   • Semantic index: 57,650 vectors
   • Emotion index: 57,650 vectors
   • Topic index: 57,650 vectors


In [ ]:
# Cell 12: FIXED Recommendation function
def recommend_similar_songs(selected_song_idx, top_k=10, weights=[0.5, 0.3, 0.2]):
    """
    Find songs similar to the selected song - FIXED VERSION
    """
    # Get query vectors for the selected song
    query_semantic = bert_norm[selected_song_idx:selected_song_idx+1]
    query_emotion = emotion_norm[selected_song_idx:selected_song_idx+1]
    query_topic = topic_norm[selected_song_idx:selected_song_idx+1]

    # Search more than needed to get diverse results
    search_k = min(200, len(df))

    # FIX: Use correct index names (remove _cpu if you didn't create them)
    semantic_scores, semantic_indices = bert_index_cpu.search(query_semantic, search_k)
    emotion_scores, emotion_indices = emotion_index_cpu.search(query_emotion, search_k)
    topic_scores, topic_indices = topic_index_cpu.search(query_topic, search_k)

    # Create dictionaries for faster lookup
    semantic_dict = {idx: score for idx, score in zip(semantic_indices[0], semantic_scores[0])}
    emotion_dict = {idx: score for idx, score in zip(emotion_indices[0], emotion_scores[0])}
    topic_dict = {idx: score for idx, score in zip(topic_indices[0], topic_scores[0])}

    # Combine scores with weights
    combined_scores = {}

    # Process all unique indices
    all_indices = set(semantic_indices[0]) | set(emotion_indices[0]) | set(topic_indices[0])

    for idx in all_indices:
        if idx == selected_song_idx:
            continue  # Skip query song

        total_score = 0
        if idx in semantic_dict:
            total_score += weights[0] * semantic_dict[idx]
        if idx in emotion_dict:
            total_score += weights[1] * emotion_dict[idx]
        if idx in topic_dict:
            total_score += weights[2] * topic_dict[idx]

        combined_scores[idx] = total_score

    # Get top K recommendations
    top_indices = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # Prepare results
    results = []
    for idx, total_score in top_indices:
        # Get individual scores
        sem_score = semantic_dict.get(idx, 0)
        emo_score = emotion_dict.get(idx, 0)
        top_score = topic_dict.get(idx, 0)

        results.append({
            'song': df.iloc[idx]['song'],
            'artist': df.iloc[idx]['artist'],
            'total_score': float(total_score),
            'semantic_score': float(sem_score),
            'emotion_score': float(emo_score),
            'topic_score': float(top_score),
            'lyrics_preview': df.iloc[idx]['text'][:150] + '...',
            'original_emotion': emotion_labels[np.argmax(emotion_embeddings[idx])] if hasattr(emotion_embeddings, '__getitem__') else 'unknown'
        })

    return results

# Test the system PROPERLY
print("🧪 TESTING RECOMMENDATION SYSTEM...")
print("="*60)

# Test with different songs to ensure they're DIFFERENT
test_indices = [1500, 2500, 3500, 4500, 5000]

for idx in test_indices:
    print(f"\n🎵 Selected song: {df.iloc[idx]['song']} by {df.iloc[idx]['artist']}")
    print("-"*40)

    recommendations = recommend_similar_songs(idx, top_k=3, weights=[0.5, 0.3, 0.2])

    if not recommendations:
        print("❌ No recommendations found!")
    else:
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec['song']} by {rec['artist']} (Score: {rec['total_score']:.3f})")

🧪 TESTING RECOMMENDATION SYSTEM...

🎵 Selected song: Hand Of Doom by Black Sabbath
----------------------------------------
1. Hand Of Doom by HIM (Score: 0.776)
2. Hand Of Doom by Slayer (Score: 0.671)
3. The Pillow by UB40 (Score: 0.646)

🎵 Selected song: Who D'king by Cheap Trick
----------------------------------------
1. Shake My Tree by Whitesnake (Score: 0.579)
2. You're The One by Tracy Chapman (Score: 0.570)
3. Hair Of The Dog by Nazareth (Score: 0.498)

🎵 Selected song: Low by Counting Crows
----------------------------------------
1. Blow It In The Wind by Chris Brown (Score: 0.522)
2. Big Fat Money by Van Halen (Score: 0.518)
3. Staying Power by Queen (Score: 0.516)

🎵 Selected song: I've Got My Love To Keep Me Warm by Doris Day
----------------------------------------
1. I've Got My Love To Keep Me Warm by Bette Midler (Score: 0.998)
2. I've Got My Love To Keep Me Warm by Ingrid Michaelson (Score: 0.793)
3. I've Got My Love To Keep Me Warm by Rod Stewart (Score: 0.793)

🎵 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 13: Save everything
import pickle

print("💾 Saving all data...")

save_data = {
    'df': df[['song', 'artist', 'text', 'processed_text']],
    'bert_embeddings': bert_embeddings,
    'emotion_embeddings': emotion_embeddings,
    'topic_embeddings': topic_embeddings,
    'bert_norm': bert_norm,
    'emotion_norm': emotion_norm,
    'topic_norm': topic_norm,
    'emotion_labels': emotion_labels,
    'topics': topics,
    'vectorizer': vectorizer,
    'lda_model': lda
}

with open('song_recommendation_system.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print("✅ Saved: song_recommendation_system.pkl")
print(f"\n🎯 SYSTEM READY! Features:")
print(f"   • {len(df):,} songs in database")
print(f"   • Semantic analysis: BERT embeddings")
print(f"   • Emotion analysis: {len(emotion_labels)} emotions")
print(f"   • Topic modeling: {len(topics)} topics")
print(f"   • FAISS CPU indexing for fast search")

💾 Saving all data...
✅ Saved: song_recommendation_system.pkl

🎯 SYSTEM READY! Features:
   • 57,650 songs in database
   • Semantic analysis: BERT embeddings
   • Emotion analysis: 7 emotions
   • Topic modeling: 15 topics
   • FAISS GPU indexing for fast search


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import faiss
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

st.set_page_config(
    page_title="Lyrics Recommender",
    page_icon="🎵",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for better styling
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        color: #1DB954;
        font-weight: bold;
        margin-bottom: 1rem;
    }
    .sub-header {
        font-size: 1.5rem;
        color: #FFFFFF;
        font-weight: 600;
        margin-top: 1rem;
        margin-bottom: 1rem;
    }
    .metric-card {
        background-color: #1E1E1E;
        padding: 1rem;
        border-radius: 10px;
        border-left: 5px solid #1DB954;
        margin-bottom: 1rem;
    }
    .song-card {
        background-color: #2D2D2D;
        padding: 1rem;
        border-radius: 10px;
        margin-bottom: 0.5rem;
        border: 1px solid #404040;
    }
    .emotion-badge {
        display: inline-block;
        padding: 0.25rem 0.75rem;
        border-radius: 15px;
        font-size: 0.8rem;
        margin-right: 0.5rem;
        margin-bottom: 0.5rem;
    }
    .score-badge {
        display: inline-block;
        padding: 0.25rem 0.5rem;
        border-radius: 5px;
        font-size: 0.9rem;
        font-weight: bold;
        margin-right: 0.5rem;
    }
    /* FIX: Reduce dropdown height */
    .stSelectbox > div > div {
        max-height: 150px !important;
        overflow-y: auto !important;
    }
    /* FIX: Make topics more visible */
    .topic-card {
        background-color: #3A3A3A;
        padding: 1rem;
        border-radius: 10px;
        margin-bottom: 0.5rem;
        border: 1px solid #505050;
    }
</style>
""", unsafe_allow_html=True)

# Load data and create FAISS indices
@st.cache_resource
def load_data():
    with open('song_recommendation_system (1).pkl', 'rb') as f:
        data = pickle.load(f)

    # Get embeddings
    bert_embeddings = data['bert_embeddings']
    emotion_embeddings = data['emotion_embeddings']
    topic_embeddings = data['topic_embeddings']

    # Normalize for cosine similarity
    def normalize(embeddings):
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        return embeddings / norms

    bert_norm = normalize(bert_embeddings).astype('float32')
    emotion_norm = normalize(emotion_embeddings).astype('float32')
    topic_norm = normalize(topic_embeddings).astype('float32')

    # Create FAISS indices
    bert_index = faiss.IndexFlatIP(bert_norm.shape[1])
    bert_index.add(bert_norm)

    emotion_index = faiss.IndexFlatIP(emotion_norm.shape[1])
    emotion_index.add(emotion_norm)

    topic_index = faiss.IndexFlatIP(topic_norm.shape[1])
    topic_index.add(topic_norm)

    return {
        'df': data['df'],
        'emotion_labels': data['emotion_labels'],
        'topics': data['topics'],
        'bert_norm': bert_norm,
        'emotion_norm': emotion_norm,
        'topic_norm': topic_norm,
        'bert_index': bert_index,
        'emotion_index': emotion_index,
        'topic_index': topic_index
    }

# Load all data
data = load_data()
df = data['df']

# Create song list for dropdown - ALL SONGS
@st.cache_data
def get_song_list():
    return [f"{row['song']} by {row['artist']}" for _, row in df.iterrows()]

song_list = get_song_list()

# Recommendation function
def recommend_similar_songs_app(selected_song_idx, top_k=10, weights=[0.5, 0.3, 0.2]):
    """Find songs similar to the selected song"""
    # Get query vectors
    query_semantic = data['bert_norm'][selected_song_idx:selected_song_idx+1]
    query_emotion = data['emotion_norm'][selected_song_idx:selected_song_idx+1]
    query_topic = data['topic_norm'][selected_song_idx:selected_song_idx+1]

    # Search more than needed
    search_k = min(200, len(df))

    # Search each index
    semantic_scores, semantic_indices = data['bert_index'].search(query_semantic, search_k)
    emotion_scores, emotion_indices = data['emotion_index'].search(query_emotion, search_k)
    topic_scores, topic_indices = data['topic_index'].search(query_topic, search_k)

    # Create dictionaries for faster lookup
    semantic_dict = {idx: score for idx, score in zip(semantic_indices[0], semantic_scores[0])}
    emotion_dict = {idx: score for idx, score in zip(emotion_indices[0], emotion_scores[0])}
    topic_dict = {idx: score for idx, score in zip(topic_indices[0], topic_scores[0])}

    # Combine scores with weights
    combined_scores = {}

    # Process all unique indices
    all_indices = set(semantic_indices[0]) | set(emotion_indices[0]) | set(topic_indices[0])

    for idx in all_indices:
        if idx == selected_song_idx:
            continue  # Skip query song

        total_score = 0
        if idx in semantic_dict:
            total_score += weights[0] * semantic_dict[idx]
        if idx in emotion_dict:
            total_score += weights[1] * emotion_dict[idx]
        if idx in topic_dict:
            total_score += weights[2] * topic_dict[idx]

        combined_scores[idx] = total_score

    # Get top K recommendations
    top_indices = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # Prepare results
    results = []
    for idx, total_score in top_indices:
        # Get individual scores
        sem_score = semantic_dict.get(idx, 0)
        emo_score = emotion_dict.get(idx, 0)
        top_score = topic_dict.get(idx, 0)

        # Get all emotion scores
        emotion_probs = data['emotion_norm'][idx]
        emotion_details = {data['emotion_labels'][i]: float(emotion_probs[i]) for i in range(len(data['emotion_labels']))}

        results.append({
            'song': df.iloc[idx]['song'],
            'artist': df.iloc[idx]['artist'],
            'total_score': float(total_score),
            'semantic_score': float(sem_score),
            'emotion_score': float(emo_score),
            'topic_score': float(top_score),
            'lyrics_preview': df.iloc[idx]['text'][:200] + '...',
            'emotion_details': emotion_details,
            'dominant_emotion': data['emotion_labels'][np.argmax(emotion_probs)]
        })

    return results

# App title
st.markdown('<h1 class="main-header">Music Recommendation System🎵</h1>', unsafe_allow_html=True)
st.markdown("**Analyze songs by Semantic Meaning + Emotional Tone + Thematic Topics**")

# Sidebar Navigation
st.sidebar.markdown("---")
app_mode = st.sidebar.radio(
    " **NAVIGATION**",
    ["🎯 Recommender System", "📊 System Overview"],
    index=0
)
st.sidebar.markdown("---")

# ===============================
# SYSTEM OVERVIEW PAGE
# ===============================
if app_mode == "📊 System Overview":
    st.markdown('<h2 class="sub-header">📊 System Overview</h2>', unsafe_allow_html=True)

    # Quick stats in columns
    col1, col2, col3, col4, col5 = st.columns(5)
    with col1:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        st.metric("🎵 Total Songs", f"{len(df):,}")
        st.markdown('</div>', unsafe_allow_html=True)

    with col2:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        st.metric("👨‍🎤 Artists", df['artist'].nunique())
        st.markdown('</div>', unsafe_allow_html=True)

    with col3:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        st.metric("😊 Emotions", len(data['emotion_labels']))
        st.markdown('</div>', unsafe_allow_html=True)

    with col4:
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        st.metric("📚 Topics", len(data['topics']))
        st.markdown('</div>', unsafe_allow_html=True)

    with col5:
        avg_words = df['text'].apply(lambda x: len(x.split())).mean()
        st.markdown('<div class="metric-card">', unsafe_allow_html=True)
        st.metric("📝 Avg Words", f"{avg_words:.0f}")
        st.markdown('</div>', unsafe_allow_html=True)

    # EDA Section
    with st.expander("📈 **Dataset Statistics & EDA**", expanded=True):
        col1, col2 = st.columns(2)

        with col1:
            # Top 20 Artists
            top_artists = df['artist'].value_counts().head(20)
            fig = px.bar(
                x=top_artists.index,
                y=top_artists.values,
                title="Top 20 Artists by Song Count",
                labels={'x': 'Artist', 'y': 'Number of Songs'},
                color=top_artists.values,
                color_continuous_scale='viridis'
            )
            fig.update_layout(height=400, template="plotly_dark")
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            # Song Length Distribution
            df['word_count'] = df['text'].apply(lambda x: len(x.split()))
            fig = px.histogram(
                df,
                x='word_count',
                nbins=50,
                title="Distribution of Song Length (Word Count)",
                labels={'word_count': 'Number of Words'},
                color_discrete_sequence=['#1DB954']
            )
            fig.update_layout(height=400, template="plotly_dark")
            st.plotly_chart(fig, use_container_width=True)

    # Emotion Analysis Section
    with st.expander("😊 **Emotion Distribution Analysis**", expanded=True):
        col1, col2 = st.columns(2)

        with col1:
            # Calculate overall emotion distribution
            overall_emotions = np.mean(data['emotion_norm'], axis=0)

            # Create emotion distribution chart
            fig = go.Figure(data=[go.Bar(
                x=data['emotion_labels'],
                y=overall_emotions,
                marker_color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']
            )])
            fig.update_layout(
                title="Overall Emotion Distribution in Dataset",
                height=400,
                template="plotly_dark",
                xaxis_title="Emotions",
                yaxis_title="Average Score"
            )
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            # Dominant emotions distribution
            dominant_emotions = []
            for i in range(len(df)):
                emotion_probs = data['emotion_norm'][i]
                dominant_idx = np.argmax(emotion_probs)
                dominant_emotions.append(data['emotion_labels'][dominant_idx])

            emotion_counts = pd.Series(dominant_emotions).value_counts()
            fig = px.pie(
                values=emotion_counts.values,
                names=emotion_counts.index,
                title="Distribution of Dominant Emotions",
                color_discrete_sequence=px.colors.qualitative.Set3
            )
            fig.update_layout(height=400, template="plotly_dark")
            st.plotly_chart(fig, use_container_width=True)

        # Emotion details in a better layout
        st.markdown("**Emotion Categories:**")
        emotion_cols = st.columns(7)
        emotion_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']

        for i, emotion in enumerate(data['emotion_labels']):
            with emotion_cols[i]:
                st.markdown(
                    f'<div class="emotion-badge" style="background-color: {emotion_colors[i]};">'
                    f'<strong>{emotion.title()}</strong><br>'
                    f'<small>Avg: {overall_emotions[i]:.3f}</small>'
                    f'</div>',
                    unsafe_allow_html=True
                )

    # Topics Section - FIXED: Use topic-card for better visibility
    with st.expander("📚 **Discovered Topics**", expanded=True):
        # Display topics in a better grid
        topics = data['topics']
        num_topics = len(topics)
        cols = st.columns(3)

        for i, (topic, words) in enumerate(topics.items()):
            with cols[i % 3]:
                # FIX: Use topic-card class instead of song-card for better visibility
                st.markdown(
                    f'<div class="topic-card">'
                    f'<strong style="color: #1DB954; font-size: 18px;">{topic}</strong><br>'
                    f'<div style="margin-top: 10px; color: #E0E0E0;">'
                    f'{" • ".join(words[:8])}'
                    f'</div>'
                    f'</div>',
                    unsafe_allow_html=True
                )

    # Dataset Sample
    with st.expander("🎵 **Dataset Sample (500 Songs)**", expanded=True):
        # Show sample songs with more information
        sample_df = df.head(500)[['song', 'artist']].copy()
        sample_df.index = range(1, len(sample_df) + 1)

        # Add search functionality
        search_term = st.text_input("🔍 Search within sample:", "")
        if search_term:
            sample_df = sample_df[
                sample_df['song'].str.contains(search_term, case=False) |
                sample_df['artist'].str.contains(search_term, case=False)
            ]

        st.dataframe(
            sample_df,
            use_container_width=True,
            height=500,
            column_config={
                "song": "Song Title",
                "artist": "Artist"
            }
        )

        # Show dataset info
        st.info(f"Showing {len(sample_df)} out of 500 sample songs. Full dataset contains {len(df):,} songs.")

    # Additional EDA Charts
    with st.expander("📊 **Advanced Dataset Analysis**", expanded=False):
        col1, col2 = st.columns(2)

        with col1:
            # Year distribution if available
            if 'year' in df.columns:
                df_year = df.dropna(subset=['year'])
                if len(df_year) > 0:
                    fig = px.histogram(
                        df_year,
                        x='year',
                        title="Song Distribution by Year",
                        nbins=30,
                        color_discrete_sequence=['#1DB954']
                    )
                    fig.update_layout(height=400, template="plotly_dark")
                    st.plotly_chart(fig, use_container_width=True)

            # Correlation between emotion dimensions
            emotion_df = pd.DataFrame(data['emotion_norm'], columns=data['emotion_labels'])
            correlation = emotion_df.corr()

            fig = go.Figure(data=go.Heatmap(
                z=correlation.values,
                x=correlation.columns,
                y=correlation.index,
                colorscale='viridis'
            ))
            fig.update_layout(
                title="Emotion Dimension Correlations",
                height=400,
                template="plotly_dark"
            )
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            # Top genres if available
            if 'genre' in df.columns:
                top_genres = df['genre'].value_counts().head(15)
                fig = px.bar(
                    x=top_genres.values,
                    y=top_genres.index,
                    orientation='h',
                    title="Top 15 Genres",
                    color=top_genres.values,
                    color_continuous_scale='thermal'
                )
                fig.update_layout(height=400, template="plotly_dark")
                st.plotly_chart(fig, use_container_width=True)

            # Embedding dimensionality info
            st.markdown("**Embedding Dimensions:**")
            st.markdown(f"- **Semantic Embeddings:** {data['bert_norm'].shape[1]} dimensions")
            st.markdown(f"- **Emotion Embeddings:** {data['emotion_norm'].shape[1]} dimensions")
            st.markdown(f"- **Topic Embeddings:** {data['topic_norm'].shape[1]} dimensions")

    with st.expander("⚙️ **Technical Details**", expanded=False):
        col1, col2 = st.columns(2)
        with col1:
            st.markdown("**Models Used:**")
            st.markdown("- **BERT:** `all-MiniLM-L6-v2` for semantic similarity")
            st.markdown("- **Emotion:** `DistilRoBERTa-base` fine-tuned on emotion detection")
            st.markdown("- **Topic Modeling:** LDA (Latent Dirichlet Allocation)")
            st.markdown("- **Similarity Search:** FAISS (Facebook AI Similarity Search)")

        with col2:
            st.markdown("**System Features:**")
            st.markdown("- Multi-dimensional recommendation (semantic + emotion + topic)")
            st.markdown("- Real-time similarity search")
            st.markdown("- Interactive visualizations")
            st.markdown("- Weighted recommendation tuning")
            st.markdown(f"- Database: {len(df):,} songs with embeddings")

# ===============================
# RECOMMENDER SYSTEM PAGE
# ===============================
else:
    st.markdown('<h2 class="sub-header">🎯 Recommender System</h2>', unsafe_allow_html=True)

    # Search and select section
    col_search, col_weights = st.columns([2, 1])

    with col_search:
        # Search box with autocomplete-like dropdown
        search_query = st.text_input(
            "🔍 **Search for a song or artist:**",
            placeholder="Type song name or artist...",
            help=f"Search through {len(df):,} songs"
        )

    with col_weights:
        # Weight controls
        with st.expander("⚖️ Adjust Weights"):
            sem_w = st.slider("Semantic", 0.0, 1.0, 0.5, 0.1)
            emo_w = st.slider("Emotion", 0.0, 1.0, 0.3, 0.1)
            top_w = st.slider("Topic", 0.0, 1.0, 0.2, 0.1)

            # Normalize
            total = sem_w + emo_w + top_w
            if total > 0:
                weights = [sem_w/total, emo_w/total, top_w/total]
            else:
                weights = [0.5, 0.3, 0.2]

    # Filter songs based on search - Show ALL songs in dropdown
    if search_query:
        filtered_options = []
        filtered_indices = []

        for i, (song, artist) in enumerate(zip(df['song'], df['artist'])):
            if search_query.lower() in song.lower() or search_query.lower() in artist.lower():
                filtered_options.append(f"{song} by {artist}")
                filtered_indices.append(i)

        if filtered_options:
            selected_option = st.selectbox(
                f"🎼 **Select a song from results ({len(filtered_options)} found):**",
                filtered_options,
                index=0,
                key="filtered_select"
            )
            selected_idx = filtered_indices[filtered_options.index(selected_option)]
        else:
            st.warning(f"No songs found matching '{search_query}'. Showing all {len(song_list):,} songs.")
            # FIX: Add info message about dropdown size
            st.info(f"💡 The dropdown shows all {len(song_list):,} songs. Use the search box above to filter.")
            selected_option = st.selectbox(
                f"🎼 **Select a song (all {len(song_list):,} songs):**",
                song_list,
                index=0,
                key="all_select"
            )
            selected_idx = song_list.index(selected_option)
    else:
        # Show ALL songs in dropdown (57650 songs) - FIX: Add info message
        st.info(f"💡 The dropdown shows all {len(song_list):,} songs. Use the search box above to filter.")
        selected_option = st.selectbox(
            f"🎼 **Select a song (all {len(song_list):,} songs):**",
            song_list,
            index=0,
            key="all_select",
            help="Scroll through the list or type in search box above to filter"
        )
        selected_idx = song_list.index(selected_option)

    # Display selected song
    st.markdown("---")
    col_song_info, col_song_lyrics = st.columns([1, 2])

    with col_song_info:
        st.markdown('<div class="song-card">', unsafe_allow_html=True)
        st.markdown(f"**🎵 Selected Song**")
        st.markdown(f"### {df.iloc[selected_idx]['song']}")
        st.markdown(f"*by {df.iloc[selected_idx]['artist']}*")
        st.markdown('</div>', unsafe_allow_html=True)

        # Emotion analysis for selected song - AS BAR CHART
        emotion_probs = data['emotion_norm'][selected_idx]

        # Create bar chart
        fig = go.Figure(data=[go.Bar(
            x=data['emotion_labels'],
            y=emotion_probs,
            marker_color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']
        )])
        fig.update_layout(
            title="Emotion Analysis",
            height=300,
            template="plotly_dark",
            xaxis_title="Emotions",
            yaxis_title="Score",
            showlegend=False
        )
        st.plotly_chart(fig, use_container_width=True)

        # Dominant emotion
        dominant_idx = np.argmax(emotion_probs)
        dominant_emotion = data['emotion_labels'][dominant_idx]
        st.markdown(f"**Dominant Emotion:** `{dominant_emotion.title()}` ({emotion_probs[dominant_idx]:.2f})")

    with col_song_lyrics:
        with st.expander("📜 **View Full Lyrics**", expanded=True):
            lyrics_text = df.iloc[selected_idx]['text']
            word_count = len(lyrics_text.split())
            st.markdown(f"*Word count: {word_count}*")
            st.text_area(
                "Lyrics",
                lyrics_text,
                height=300,
                label_visibility="collapsed"
            )

    # Recommendation button
    if st.button("🚀 **Find Similar Songs**", type="primary", use_container_width=True):
        with st.spinner(f"🔍 Searching through {len(df):,} songs..."):
            recommendations = recommend_similar_songs_app(
                selected_idx,
                top_k=10,
                weights=weights
            )

            # Store in session state
            st.session_state.recommendations = recommendations
            st.session_state.selected_idx = selected_idx
            st.session_state.selected_song = f"{df.iloc[selected_idx]['song']} by {df.iloc[selected_idx]['artist']}"
            st.session_state.weights = weights

    # Display recommendations if available
    if 'recommendations' in st.session_state:
        st.markdown("---")
        st.markdown(f'<h3 class="sub-header">🎯 Similar to: {st.session_state.selected_song}</h3>', unsafe_allow_html=True)

        # Show current weights
        st.markdown(f"**Weights:** Semantic ({st.session_state.weights[0]:.0%}) | Emotion ({st.session_state.weights[1]:.0%}) | Topic ({st.session_state.weights[2]:.0%})")

        # Emotion distribution of recommendations
        st.markdown("**😊 Emotion Distribution in Recommendations:**")
        emotion_counts = {}
        for rec in st.session_state.recommendations:
            emotion = rec['dominant_emotion']
            emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1

        # Create pie chart
        fig = go.Figure(data=[go.Pie(
            labels=list(emotion_counts.keys()),
            values=list(emotion_counts.values()),
            hole=0.3,
            marker_colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']
        )])
        fig.update_layout(
            height=300,
            showlegend=True,
            margin=dict(l=0, r=0, t=30, b=0),
            template="plotly_dark"
        )
        st.plotly_chart(fig, use_container_width=True)

        # Display each recommendation
        st.markdown("**🎵 Recommended Songs:**")

        for i, rec in enumerate(st.session_state.recommendations, 1):
            # Create a card for each recommendation
            with st.container():
                st.markdown('<div class="song-card">', unsafe_allow_html=True)

                # Header with score
                col_rank, col_title, col_score = st.columns([1, 4, 2])
                with col_rank:
                    st.markdown(f"### #{i}")
                with col_title:
                    st.markdown(f"**{rec['song']}**")
                    st.markdown(f"*by {rec['artist']}*")
                with col_score:
                    score_color = "#1DB954" if rec['total_score'] > 0.7 else "#FFA500" if rec['total_score'] > 0.5 else "#FF6B6B"
                    st.markdown(f'<div class="score-badge" style="background-color: {score_color}; color: white;">Overall: {rec["total_score"]:.3f}</div>', unsafe_allow_html=True)

                # Detailed scores
                col_sem, col_emo, col_top = st.columns(3)
                with col_sem:
                    st.metric("Semantic", f"{rec['semantic_score']:.3f}")
                with col_emo:
                    st.metric("Emotion", f"{rec['emotion_score']:.3f}")
                with col_top:
                    st.metric("Topic", f"{rec['topic_score']:.3f}")

                # Emotion details as mini bar chart
                st.markdown("**Emotion Breakdown:**")

                # Create mini horizontal bar chart
                emotions = list(rec['emotion_details'].keys())
                scores = list(rec['emotion_details'].values())

                fig_mini = go.Figure(data=[go.Bar(
                    y=emotions,
                    x=scores,
                    orientation='h',
                    marker_color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8']
                )])
                fig_mini.update_layout(
                    height=150,
                    margin=dict(l=0, r=0, t=0, b=0),
                    showlegend=False,
                    xaxis=dict(range=[0, 1]),
                    template="plotly_dark"
                )
                st.plotly_chart(fig_mini, use_container_width=True)

                # Lyrics preview
                with st.expander("📜 View Lyrics Preview"):
                    st.write(rec['lyrics_preview'])

                st.markdown('</div>', unsafe_allow_html=True)
                st.markdown("<br>", unsafe_allow_html=True)

# Footer
st.markdown("---")
st.markdown(
    """
    <div style="text-align: center; color: #888;">
        <small>🎵 Multi-Dimensional Lyrics Recommender System •
        Semantic Analysis + Emotion Detection + Topic Modeling •
        Powered by BERT, FAISS & Streamlit •
        Database: {:,} songs</small>
    </div>
    """.format(len(df)),
    unsafe_allow_html=True
)

Writing app.py


In [ ]:
!pip install -q transformers torch scikit-learn pandas numpy matplotlib seaborn plotly wordcloud nltk streamlit pyngrok sentence-transformers faiss-cpu tqdm --quiet
!pip install pyngrok streamlit --quiet
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.6 MB/s eta 0:00:00


In [ ]:

import subprocess
import threading
from pyngrok import ngrok
import time
import os

os.system("pkill -9 streamlit")
os.system("pkill -9 ngrok")
time.sleep(3)

# Start App
# ⚠️ REPLACE WITH YOUR TOKEN
NGROK_AUTH_TOKEN = "37CvMET4OOF6PIxviTech4KH3Hy_5YwMk8ubbAw2LTLVQ3phd"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
public_url = ngrok.connect(8501)
print(f"🚀 APP is live at: {public_url}")

🚀 APP is live at: NgrokTunnel: "https://d6d0034737cf.ngrok-free.app" -> "http://localhost:8501"
